In [1]:
# Importación de librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Carga del dataset

df_financiero = pd.read_excel(
    "../data/Bronze/informe_resultados_exito.xlsx"
)

df_financiero["fecha"] = pd.to_datetime(
    df_financiero["fecha"]
)

df_financiero = (
    df_financiero
    .sort_values("fecha")
    .reset_index(drop=True)
)

df_financiero.head()

,año,trimestre,fecha,ingresos,utilidad_bruta,ebitda_recurrente,resultado_neto
0,2020,1,2020-03-31,4052431,1001122,262832,21987
1,2020,2,2020-06-30,3688456,911875,299143,12787
2,2020,3,2020-09-30,3649939,901871,249457,51814
3,2020,4,2020-12-31,4345013,1142061,460429,144284
4,2021,1,2021-03-31,3819172,1016535,306694,84957


In [3]:
# Verificación de dimensiones

print("Filas:", df_financiero.shape[0])
print("Columnas:", df_financiero.shape[1])

df_financiero.shape

Filas: 24
Columnas: 7


(24, 7)

In [4]:
# Definición de las variables objetivo

variables_objetivo = [
    "ingresos",
    "utilidad_bruta",
    "ebitda_recurrente",
    "resultado_neto"
]

In [5]:
# Verificación de variables

df_financiero[
    ["fecha", "año", "trimestre"] + variables_objetivo
].head()

,fecha,año,trimestre,ingresos,utilidad_bruta,ebitda_recurrente,resultado_neto
0,2020-03-31,2020,1,4052431,1001122,262832,21987
1,2020-06-30,2020,2,3688456,911875,299143,12787
2,2020-09-30,2020,3,3649939,901871,249457,51814
3,2020-12-31,2020,4,4345013,1142061,460429,144284
4,2021-03-31,2021,1,3819172,1016535,306694,84957


In [6]:
# Variables temporales

# Indice temporal

df_features = df_financiero.copy()

df_features["t"] = np.arange(
    1,
    len(df_features) + 1
)

df_features[
    ["fecha", "año", "trimestre", "t"]
].head(10)

,fecha,año,trimestre,t
0,2020-03-31,2020,1,1
1,2020-06-30,2020,2,2
2,2020-09-30,2020,3,3
3,2020-12-31,2020,4,4
4,2021-03-31,2021,1,5
5,2021-06-30,2021,2,6
6,2021-09-30,2021,3,7
7,2021-12-31,2021,4,8
8,2022-03-31,2022,1,9
9,2022-06-30,2022,2,10


In [7]:
# Creación de variables rezagadas (lags: Un lag significa utilizar información de períodos anteriores para intentar pronosticar el período actual)

# Lag de un trimestre

for variable in variables_objetivo:
    df_features[f"{variable}_lag1"] = (
        df_features[variable].shift(1)
    )

In [8]:
# Lag de 4 trimestres
#Como tenemos una posible estacionalidad anual, es especialmente importante utilizar el valor del mismo trimestre del año anterior.

for variable in variables_objetivo:
    df_features[f"{variable}_lag4"] = (
        df_features[variable].shift(4)
    )

In [9]:
# Revición de lags

columnas_lags = [
    "fecha",
    "trimestre",
    "ingresos",
    "ingresos_lag1",
    "ingresos_lag4",
    "utilidad_bruta",
    "utilidad_bruta_lag1",
    "utilidad_bruta_lag4",
    "ebitda_recurrente",
    "ebitda_recurrente_lag1",
    "ebitda_recurrente_lag4",
    "resultado_neto",
    "resultado_neto_lag1",
    "resultado_neto_lag4"
]

df_features[columnas_lags].head(10)

,fecha,trimestre,ingresos,ingresos_lag1,ingresos_lag4,utilidad_bruta,utilidad_bruta_lag1,utilidad_bruta_lag4,ebitda_recurrente,ebitda_recurrente_lag1,ebitda_recurrente_lag4,resultado_neto,resultado_neto_lag1,resultado_neto_lag4
0,2020-03-31,1,4052431,NaN,NaN,1001122,NaN,NaN,262832,NaN,NaN,21987,NaN,NaN
1,2020-06-30,2,3688456,4052431.0,NaN,911875,1001122.0,NaN,299143,262832.0,NaN,12787,21987.0,NaN
2,2020-09-30,3,3649939,3688456.0,NaN,901871,911875.0,NaN,249457,299143.0,NaN,51814,12787.0,NaN
3,2020-12-31,4,4345013,3649939.0,NaN,1142061,901871.0,NaN,460429,249457.0,NaN,144284,51814.0,NaN
4,2021-03-31,1,3819172,4345013.0,4052431.0,1016535,1142061.0,1001122.0,306694,460429.0,262832.0,84957,144284.0,21987.0
5,2021-06-30,2,3696687,3819172.0,3688456.0,959633,1016535.0,911875.0,306557,306694.0,299143.0,50744,84957.0,12787.0
6,2021-09-30,3,4163857,3696687.0,3649939.0,1061678,959633.0,901871.0,353514,306557.0,249457.0,126315,50744.0,51814.0
7,2021-12-31,4,5242669,4163857.0,4345013.0,1395683,1061678.0,1142061.0,568638,353514.0,460429.0,212665,126315.0,144284.0
8,2022-03-31,1,4601967,5242669.0,3819172.0,1174498,1395683.0,1016535.0,355163,568638.0,306694.0,64539,212665.0,84957.0
9,2022-06-30,2,4717215,4601967.0,3696687.0,1186872,1174498.0,959633.0,370917,355163.0,306557.0,62264,64539.0,50744.0


In [10]:
# Media movil histórica de 4 trimestres

for variable in variables_objetivo:
    df_features[f"{variable}_rolling_mean_4"] = (
        df_features[variable]
        .shift(1)
        .rolling(window=4)
        .mean()
    )

In [11]:
# Desviación estandar histórica de 4 trimestres

for variable in variables_objetivo:
    df_features[f"{variable}_rolling_std_4"] = (
        df_features[variable]
        .shift(1)
        .rolling(window=4)
        .std()
    )

In [12]:
# Crecimiento YoY basado en lag 4

variables_crecimiento = [
    "ingresos",
    "utilidad_bruta",
    "ebitda_recurrente"
]

for variable in variables_crecimiento:

    df_features[f"{variable}_yoy_pct"] = (
        (
            df_features[variable] /
            df_features[f"{variable}_lag4"]
        ) - 1
    ) * 100

In [13]:
# Variación absoluta del resultado neto

df_features["resultado_neto_yoy_diff"] = (
    df_features["resultado_neto"] -
    df_features["resultado_neto_lag4"]
)

In [14]:
# Variables estacionales
# Variables dummy de trimestre

df_features = pd.get_dummies(
    df_features,
    columns=["trimestre"],
    prefix="T",
    dtype=int
)

In [17]:
df_features.shape


(24, 31)

In [18]:
df_features.columns.tolist()

['año',
 'fecha',
 'ingresos',
 'utilidad_bruta',
 'ebitda_recurrente',
 'resultado_neto',
 't',
 'ingresos_lag1',
 'utilidad_bruta_lag1',
 'ebitda_recurrente_lag1',
 'resultado_neto_lag1',
 'ingresos_lag4',
 'utilidad_bruta_lag4',
 'ebitda_recurrente_lag4',
 'resultado_neto_lag4',
 'ingresos_rolling_mean_4',
 'utilidad_bruta_rolling_mean_4',
 'ebitda_recurrente_rolling_mean_4',
 'resultado_neto_rolling_mean_4',
 'ingresos_rolling_std_4',
 'utilidad_bruta_rolling_std_4',
 'ebitda_recurrente_rolling_std_4',
 'resultado_neto_rolling_std_4',
 'ingresos_yoy_pct',
 'utilidad_bruta_yoy_pct',
 'ebitda_recurrente_yoy_pct',
 'resultado_neto_yoy_diff',
 'T_1',
 'T_2',
 'T_3',
 'T_4']

In [19]:
# Revisión de faltantes

df_features.isna().sum().sort_values(
    ascending=False
)

utilidad_bruta_lag4                 4
ebitda_recurrente_lag4              4
resultado_neto_lag4                 4
ingresos_lag4                       4
ebitda_recurrente_yoy_pct           4
utilidad_bruta_yoy_pct              4
ingresos_yoy_pct                    4
resultado_neto_rolling_std_4        4
ebitda_recurrente_rolling_std_4     4
utilidad_bruta_rolling_std_4        4
ingresos_rolling_std_4              4
resultado_neto_rolling_mean_4       4
ebitda_recurrente_rolling_mean_4    4
utilidad_bruta_rolling_mean_4       4
ingresos_rolling_mean_4             4
resultado_neto_yoy_diff             4
resultado_neto_lag1                 1
utilidad_bruta_lag1                 1
ebitda_recurrente_lag1              1
ingresos_lag1                       1
resultado_neto                      0
t                                   0
ebitda_recurrente                   0
utilidad_bruta                      0
año                                 0
fecha                               0
ingresos    

In [20]:
df_features[
    [
        "fecha",
        "ingresos",
        "ingresos_lag1",
        "ingresos_lag4",
        "ingresos_rolling_mean_4",
        "ingresos_yoy_pct"
    ]
].head(10)

,fecha,ingresos,ingresos_lag1,ingresos_lag4,ingresos_rolling_mean_4,ingresos_yoy_pct
0,2020-03-31,4052431,NaN,NaN,NaN,NaN
1,2020-06-30,3688456,4052431.0,NaN,NaN,NaN
2,2020-09-30,3649939,3688456.0,NaN,NaN,NaN
3,2020-12-31,4345013,3649939.0,NaN,NaN,NaN
4,2021-03-31,3819172,4345013.0,4052431.0,3933959.75,-5.756026
5,2021-06-30,3696687,3819172.0,3688456.0,3875645.00,0.223156
6,2021-09-30,4163857,3696687.0,3649939.0,3877702.75,14.080181
7,2021-12-31,5242669,4163857.0,4345013.0,4006182.25,20.659455
8,2022-03-31,4601967,5242669.0,3819172.0,4230596.25,20.496458
9,2022-06-30,4717215,4601967.0,3696687.0,4426295.00,27.606557


In [21]:
# Eliminar las primeras observaciones que contienen valores nulos debido a los lags y medias móviles

df_model = df_features.dropna().copy()

df_model = (
    df_model
    .sort_values("fecha")
    .reset_index(drop=True)
)

print("Observaciones originales:", len(df_features))
print("Observaciones para modelamiento:", len(df_model))

Observaciones originales: 24
Observaciones para modelamiento: 20


In [22]:
# Verificar que no exista leakage

df_model[
    [
        "fecha",
        "ingresos",
        "ingresos_lag1",
        "ingresos_lag4",
        "ingresos_rolling_mean_4",
        "ingresos_yoy_pct"
    ]
].head(10)

,fecha,ingresos,ingresos_lag1,ingresos_lag4,ingresos_rolling_mean_4,ingresos_yoy_pct
0,2021-03-31,3819172,4345013.0,4052431.0,3933959.75,-5.756026
1,2021-06-30,3696687,3819172.0,3688456.0,3875645.00,0.223156
2,2021-09-30,4163857,3696687.0,3649939.0,3877702.75,14.080181
3,2021-12-31,5242669,4163857.0,4345013.0,4006182.25,20.659455
4,2022-03-31,4601967,5242669.0,3819172.0,4230596.25,20.496458
5,2022-06-30,4717215,4601967.0,3696687.0,4426295.00,27.606557
6,2022-09-30,5103845,4717215.0,4163857.0,4681427.00,22.574935
7,2022-12-31,6196646,5103845.0,5242669.0,4916424.00,18.196400
8,2023-03-31,5456154,6196646.0,4601967.0,5154918.25,18.561346
9,2023-06-30,5119120,5456154.0,4717215.0,5368465.00,8.519964


Ingresos actuales       → trimestre que queremos modelar

Ingresos lag1            → trimestre anterior

Ingresos lag4            → mismo trimestre año anterior

Rolling mean 4           → promedio de los cuatro trimestres anteriores

In [23]:
# Validación del dataset

df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   año                               20 non-null     int64         
 1   fecha                             20 non-null     datetime64[us]
 2   ingresos                          20 non-null     int64         
 3   utilidad_bruta                    20 non-null     int64         
 4   ebitda_recurrente                 20 non-null     int64         
 5   resultado_neto                    20 non-null     int64         
 6   t                                 20 non-null     int64         
 7   ingresos_lag1                     20 non-null     float64       
 8   utilidad_bruta_lag1               20 non-null     float64       
 9   ebitda_recurrente_lag1            20 non-null     float64       
 10  resultado_neto_lag1               20 non-null     float64      

In [24]:
df_model.head()

,año,fecha,ingresos,utilidad_bruta,ebitda_recurrente,resultado_neto,t,ingresos_lag1,utilidad_bruta_lag1,ebitda_recurrente_lag1,...,ebitda_recurrente_rolling_std_4,resultado_neto_rolling_std_4,ingresos_yoy_pct,utilidad_bruta_yoy_pct,ebitda_recurrente_yoy_pct,resultado_neto_yoy_diff,T_1,T_2,T_3,T_4
0,2021,2021-03-31,3819172,1016535,306694,84957,5,4345013.0,1142061.0,460429.0,...,97268.104969,60066.709288,-5.756026,1.539573,16.688227,62970.0,1,0,0,0
1,2021,2021-06-30,3696687,959633,306557,50744,6,3819172.0,1016535.0,306694.0,...,91268.194155,55671.603261,0.223156,5.237341,2.478413,37957.0,0,1,0,0
2,2021,2021-09-30,4163857,1061678,353514,126315,7,3696687.0,959633.0,306557.0,...,90533.939461,43865.569401,14.080181,17.719496,41.713402,74501.0,0,0,1,0
3,2021,2021-12-31,5242669,1395683,568638,212665,8,4163857.0,1061678.0,353514.0,...,72536.744052,42016.166595,20.659455,22.207395,23.501778,68381.0,0,0,0,1
4,2022,2022-03-31,4601967,1174498,355163,64539,9,5242669.0,1395683.0,568638.0,...,125158.744564,69866.572586,20.496458,15.539357,15.803700,-20418.0,1,0,0,0


In [25]:
# Separación de variables objetivo y variables predictoras

variables_objetivo

['ingresos', 'utilidad_bruta', 'ebitda_recurrente', 'resultado_neto']

In [26]:
variables_explicativas = [
    columna
    for columna in df_model.columns
    if columna not in variables_objetivo
    and columna not in ["fecha"]
]

In [27]:
variables_explicativas

['año',
 't',
 'ingresos_lag1',
 'utilidad_bruta_lag1',
 'ebitda_recurrente_lag1',
 'resultado_neto_lag1',
 'ingresos_lag4',
 'utilidad_bruta_lag4',
 'ebitda_recurrente_lag4',
 'resultado_neto_lag4',
 'ingresos_rolling_mean_4',
 'utilidad_bruta_rolling_mean_4',
 'ebitda_recurrente_rolling_mean_4',
 'resultado_neto_rolling_mean_4',
 'ingresos_rolling_std_4',
 'utilidad_bruta_rolling_std_4',
 'ebitda_recurrente_rolling_std_4',
 'resultado_neto_rolling_std_4',
 'ingresos_yoy_pct',
 'utilidad_bruta_yoy_pct',
 'ebitda_recurrente_yoy_pct',
 'resultado_neto_yoy_diff',
 'T_1',
 'T_2',
 'T_3',
 'T_4']

In [29]:
df_check = pd.read_csv(
    ruta_features,
    parse_dates=["fecha"]
)

print("Dimensiones:", df_check.shape)
print(
    "Valores nulos:",
    df_check.isna().sum().sum()
)

df_check.head()

Dimensiones: (20, 31)
Valores nulos: 0


,año,fecha,ingresos,utilidad_bruta,ebitda_recurrente,resultado_neto,t,ingresos_lag1,utilidad_bruta_lag1,ebitda_recurrente_lag1,...,ebitda_recurrente_rolling_std_4,resultado_neto_rolling_std_4,ingresos_yoy_pct,utilidad_bruta_yoy_pct,ebitda_recurrente_yoy_pct,resultado_neto_yoy_diff,T_1,T_2,T_3,T_4
0,2021,2021-03-31,3819172,1016535,306694,84957,5,4345013.0,1142061.0,460429.0,...,97268.104969,60066.709288,-5.756026,1.539573,16.688227,62970.0,1,0,0,0
1,2021,2021-06-30,3696687,959633,306557,50744,6,3819172.0,1016535.0,306694.0,...,91268.194155,55671.603261,0.223156,5.237341,2.478413,37957.0,0,1,0,0
2,2021,2021-09-30,4163857,1061678,353514,126315,7,3696687.0,959633.0,306557.0,...,90533.939461,43865.569401,14.080181,17.719496,41.713402,74501.0,0,0,1,0
3,2021,2021-12-31,5242669,1395683,568638,212665,8,4163857.0,1061678.0,353514.0,...,72536.744052,42016.166595,20.659455,22.207395,23.501778,68381.0,0,0,0,1
4,2022,2022-03-31,4601967,1174498,355163,64539,9,5242669.0,1395683.0,568638.0,...,125158.744564,69866.572586,20.496458,15.539357,15.803700,-20418.0,1,0,0,0


In [30]:
# Guardado del dataset final para modelamiento

ruta_features = (
    "../data/Silver/"
    "grupo_exito_features.csv"
)

df_model.to_csv(
    ruta_features,
    index=False
)

print(
    f"Dataset de features guardado en: "
    f"{ruta_features}"
)

Dataset de features guardado en: ../data/Silver/grupo_exito_features.csv
